# Vanilla attention

Implement the core attention mechanism used in Transformers.

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

### Signature
```python
def scaled_dot_product_attention(
    Q: torch.Tensor,  # (batch, seq_q, d_k)
    K: torch.Tensor,  # (batch, seq_k, d_k)
    V: torch.Tensor,  # (batch, seq_k, d_v)
) -> torch.Tensor:   # (batch, seq_q, d_v)
    ...
```

In [1]:
def scaled_dot_product_attention(Q, K, V):
    d_k = Q.shape[-1]
    return torch.matmul(torch.softmax(torch.matmul(Q, K.transpose(1,2))/torch.sqrt(d_k), dim=-1), V)

# Multi-head attention

Implement **Multi-Head Attention** from scratch — the core building block of the Transformer.

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h) W^O$$
$$\text{head}_i = \text{Attention}(Q W_i^Q,\; K W_i^K,\; V W_i^V)$$

### Signature
```python
class MultiHeadAttention:
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, Q, K, V) -> torch.Tensor: ...
```

### Requirements
- Use `nn.Linear(d_model, d_model)` for `self.W_q`, `self.W_k`, `self.W_v`, `self.W_o`
- `d_k = d_model // num_heads` per head
- `forward(Q, K, V)`: Q is `(B, seq_q, d_model)`, K/V are `(B, seq_k, d_model)`
- Must support **cross-attention** (`seq_q != seq_k`)
- Do **NOT** use `torch.nn.MultiheadAttention`
- You **may** use `torch.softmax` and `torch.matmul`

### Steps
1. Project: `q = self.W_q(Q)`, `k = self.W_k(K)`, `v = self.W_v(V)`
2. Reshape to `(B, num_heads, seq, d_k)`
3. Scaled dot-product attention per head
4. Concat heads → `(B, seq_q, d_model)`
5. Output projection: `self.W_o(concat)`

In [3]:
class MultiHeadAttention:
    def __init__(self, d_model, num_heads):
        self.W_q = torch.nn.Linear(d_model, d_model)
        self.W_k = torch.nn.Linear(d_model, d_model)
        self.W_v = torch.nn.Linear(d_model, d_model)
        self.W_o = torch.nn.Linear(d_model, d_model)
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

    def forward(self, Q, K, V):
        q = self.W_q(Q) #[batch, n_seq, d_model]
        k = self.W_k(K)
        v = self.W_v(V)

        n_batch, q_n_seq, dim = Q.shape
        k_n_seq = K.shape[1]
        
        q = q.reshape(n_batch, q_n_seq, -1, self.d_k).transpose(1,2)
        k = k.reshape(n_batch, k_n_seq, -1, self.d_k).transpose(1,2)
        v = k.reshape(n_batch, k_n_seq, -1, self.d_k).transpose(1,2)

        attn = torch.matmul(q, k.transpose(2,3))/math.sqrt(self.d_k)
        attn = torch.softmax(attn, dim=-1)
        out = torch.matmul(attn, v)
        out = out.transpose(1,2).reshape(n_batch, q_n_seq, self.d_model)
        out = self.W_o(out)
        return out

# Grouped Query Attention (GQA)

Implement **Grouped Query Attention** — used in LLaMA 2, Mistral, etc. to reduce KV cache size.

Like MHA, but with **fewer KV heads** than Q heads. Each group of Q heads shares the same K/V head.

### Signature
```python
class GroupQueryAttention:
    def __init__(self, d_model: int, num_heads: int, num_kv_heads: int): ...
    def forward(self, x) -> torch.Tensor:  # self-attention
```

### Requirements
- `self.W_q`: `nn.Linear(d_model, d_model)` — full Q projection
- `self.W_k`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced K projection
- `self.W_v`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced V projection
- `self.W_o`: `nn.Linear(d_model, d_model)` — output projection
- `d_k = d_model // num_heads`
- Expand KV heads with `repeat_interleave` to match Q heads
- When `num_kv_heads == num_heads`, should behave like standard MHA

In [4]:
class GroupQueryAttention:
    def __init__(self, d_model, num_heads, num_kv_heads):
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.d_k = d_model // num_heads

        self.W_k = nn.Linear(d_model, num_kv_heads*self.d_k)
        self.W_v = nn.Linear(d_model, num_kv_heads*self.d_k)
        self.W_q = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x):
        k = self.W_k(x)
        v = self.W_v(x)
        q = self.W_q(x)

        batch, seq_k, _ = k.shape
        _, seq_q, d_model = q.shape
        
        k = k.reshape(batch, seq_k, self.num_kv_heads, self.d_k).transpose(1,2) 
        v = v.reshape(batch, seq_k, self.num_kv_heads, self.d_k).transpose(1,2)
        q = q.reshape(batch, seq_q, self.num_heads, self.d_k).transpose(1,2) 

        factor = self.num_heads//self.num_kv_heads
        k = torch.repeat_interleave(k, factor, dim=1)
        v = torch.repeat_interleave(v, factor, dim=1)

        a = torch.matmul(q,k.transpose(-1,-2))/math.sqrt(self.d_k)
        a = torch.softmax(a, dim=-1)
        o = torch.matmul(a, v)
        o = o.transpose(1,2).reshape(batch, seq_q, d_model)
        o = self.W_o(o)
        return o


## Flash Attention (Tiled)

Implement **tiled attention with online softmax** — the core idea behind Flash Attention.

### Signature
```python
def flash_attention(Q, K, V, block_size=32) -> Tensor:
    # Q, K, V: (B, S, D)
    # Returns: (B, S, D) — same as standard attention
```

### Key Insight
Instead of materializing the full S×S attention matrix, process in blocks:
1. For each Q-block, iterate over K/V blocks
2. Use **online softmax**: track running `max` and `sum`
3. Rescale accumulator when max changes: `acc *= exp(old_max - new_max)`
4. Final: `output = acc / row_sum`

Must give **identical** results to standard softmax attention.